In [1]:
!pip install arch neuralforecast pytorch-lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 982.9/982.9 kB 11.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.3/331.3 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.7/348.7 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 4.1 MB/s eta 0:00:00


In [2]:
import glob
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import neuralforecast
from neuralforecast import NeuralForecast
from neuralforecast.models import TFT, LSTM, NHITS, TiDE, NBEATSx, TSMixerx, KAN
from pytorch_lightning.loggers import CSVLogger
import inspect

print("NeuralForecast Version:", neuralforecast.__version__)

# 1. Load Data
dl_path = '/kaggle/input/datasets/ferastoro/ai-stocks-crypto-and-ihsg/data_clean/dataset_deep_learning.csv'
df = pd.read_csv(dl_path)

tvp_path = '/kaggle/input/datasets/ferastoro/ai-stocks-crypto-and-ihsg/data_clean/tvp_var_dynamic_features.csv'
if not os.path.exists(tvp_path):
    tvp_path = 'data_clean/tvp_var_dynamic_features.csv'
df_tvp = pd.read_csv(tvp_path)

# --- PENGAMANAN DATASET (WAJIB VOLATILITAS) ---
if df_tvp['net_spill_saham_tsla'].min() > -50:
    raise ValueError("❌ DATASET MASIH VERSI LAMA (RETURN). HARAP UPDATE DATASET DI KAGGLE KE VERSI LOG-VOLATILITAS!")
print("✅ Dataset aman! Menggunakan versi Volatilitas terbaru.")
# ----------------------------------------------

df_tvp['Date'] = pd.to_datetime(df_tvp['Date'])
df['Date'] = pd.to_datetime(df['Date'])

df = pd.merge(df, df_tvp, on='Date', how='left')
tvp_cols = [c for c in df_tvp.columns if c != 'Date']
df[tvp_cols] = df[tvp_cols].ffill()
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'Baris setelah merge TVP-VAR: {len(df)}')
assert len(df) == 1676, "Total baris harus 1676!"

df['volatility_btc_idr'] = df['ret_crypto_btc_idr_indodax']**2

# 2. Exog Cols FINAL (20 Fitur)
exog_cols = [
    'ret_saham_nvda', 'ret_saham_msft', 'ret_saham_tsla', 'ret_saham_googl', 'ret_saham_meta',
    'ret_crypto_btc_usd', 'ret_crypto_eth_usd', 'crypto_btc_usd_weekend_return',
    'spillover_ai_to_btcidr', 'tvp_tsi',
    'net_spill_saham_nvda', 'net_spill_saham_msft', 'net_spill_saham_tsla', 'net_spill_saham_googl', 'net_spill_saham_meta',
    'net_spill_crypto_btc_usd', 'net_spill_crypto_eth_usd',
    'gap_days', 'is_monday', 'is_ihsg_holiday'
]

print(f"Jumlah exog_cols: {len(exog_cols)}")
assert len(exog_cols) == 20, "Jumlah kolom eksogen harus 20!"
assert 'net_spill_crypto_btc_idr_indodax' not in exog_cols, "ADA LEAKAGE!"
assert all(c in df.columns for c in exog_cols), "Ada exog yang tidak ditemukan di DataFrame!"

# 3. Bentuk Dataset Nixtla
nf_df = pd.DataFrame()
nf_df['ds'] = pd.date_range(start='2020-01-01', periods=len(df), freq='D')
nf_df['unique_id'] = 'BTC_IDR'
nf_df['y'] = df['volatility_btc_idr'].values.astype('float32')

for col in exog_cols:
    nf_df[col] = df[col].values.astype('float32')

nf_df['real_date'] = df['Date'].values
df.set_index('Date', inplace=True)

# Split (70/10/20)
N = len(nf_df)
n_test = int(N * 0.20)
n_val = int(N * 0.10)
n_train = N - n_test - n_val

df_train = nf_df.iloc[:n_train].copy()
df_val = nf_df.iloc[n_train:n_train+n_val].copy()
df_test = nf_df.iloc[n_train+n_val:].copy()

print(f"Total data: {N} | Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")


NeuralForecast Version: 3.3.0
✅ Dataset aman! Menggunakan versi Volatilitas terbaru.
Baris setelah merge TVP-VAR: 1676
Jumlah exog_cols: 20
Total data: 1676 | Train: 1174 | Val: 167 | Test: 335


In [7]:
# ==========================================
# TRAINING FINAL 7 MODEL DL (V4) DENGAN AUTO-RESUME
# ==========================================
import os
import pandas as pd
import glob
from pytorch_lightning.loggers import CSVLogger
from neuralforecast import NeuralForecast
from neuralforecast.models import TFT, LSTM, NHITS, TiDE, NBEATSx, TSMixerx, KAN

# PASTIKAN ANDA MENGGANTI NILAI DI BAWAH INI BERDASARKAN HASIL EKSPERIMEN SEL 2
CHOSEN_PATIENCE = 50 

H = 10
INPUT_SIZE = 5*H
MAX_STEPS = 2000
VAL_CHECK_STEPS = 50
SEED = 42

models_list = [
    TFT(h=H, input_size=INPUT_SIZE, hist_exog_list=exog_cols, max_steps=MAX_STEPS, val_check_steps=VAL_CHECK_STEPS, early_stop_patience_steps=CHOSEN_PATIENCE, random_seed=SEED, scaler_type='standard'),
    LSTM(h=H, input_size=INPUT_SIZE, hist_exog_list=exog_cols, max_steps=MAX_STEPS, val_check_steps=VAL_CHECK_STEPS, early_stop_patience_steps=CHOSEN_PATIENCE, random_seed=SEED, scaler_type='standard'),
    NHITS(h=H, input_size=INPUT_SIZE, hist_exog_list=exog_cols, max_steps=MAX_STEPS, val_check_steps=VAL_CHECK_STEPS, early_stop_patience_steps=CHOSEN_PATIENCE, random_seed=SEED, scaler_type='standard'),
    TiDE(h=H, input_size=INPUT_SIZE, hist_exog_list=exog_cols, max_steps=MAX_STEPS, val_check_steps=VAL_CHECK_STEPS, early_stop_patience_steps=CHOSEN_PATIENCE, random_seed=SEED, scaler_type='standard'),
    NBEATSx(h=H, input_size=INPUT_SIZE, hist_exog_list=exog_cols, max_steps=MAX_STEPS, val_check_steps=VAL_CHECK_STEPS, early_stop_patience_steps=CHOSEN_PATIENCE, random_seed=SEED, scaler_type='standard'),
    TSMixerx(h=H, input_size=INPUT_SIZE, hist_exog_list=exog_cols, n_series=1, max_steps=MAX_STEPS, val_check_steps=VAL_CHECK_STEPS, early_stop_patience_steps=CHOSEN_PATIENCE, random_seed=SEED, scaler_type='standard'),
    KAN(h=H, input_size=INPUT_SIZE, hist_exog_list=exog_cols, max_steps=MAX_STEPS, val_check_steps=VAL_CHECK_STEPS, early_stop_patience_steps=CHOSEN_PATIENCE, random_seed=SEED, scaler_type='standard')
]

cv_file_path = '/kaggle/working/CV_Predictions_V4.csv'

# CEK: APAKAH SUDAH SELESAI SEBELUMNYA?
if os.path.exists(cv_file_path):
    print(f"✅ MENDETEKSI FILE {cv_file_path} SUDAH ADA!")
    print("Melewati proses training ulang... Langsung memuat data ke memori.")
    cv_df = pd.read_csv(cv_file_path)
    cv_df['ds'] = pd.to_datetime(cv_df['ds'])
    if 'real_date' in cv_df.columns:
        cv_df['real_date'] = pd.to_datetime(cv_df['real_date'])
        
else:
    print(f"Memulai training 7 Model DL (max_steps=2000, patience={CHOSEN_PATIENCE})...")
    for m in models_list:
        m_name = m.__class__.__name__
        logger = CSVLogger(save_dir='/kaggle/working/logs', name=f'{m_name}_V4')
        
        m.trainer_kwargs = {
            'logger': logger, 
            'enable_checkpointing': True, 
            'devices': 1,
            'max_epochs': 2000,
            'max_steps': 2000
        }

    nf = NeuralForecast(models=models_list, freq='D')

    print("Memulai fit (Training)...")
    nf.fit(df=df_train.drop(columns=['real_date']), val_size=n_val)

    print("Training selesai. Memulai Cross-Validation (Prediksi)...")
    cv_df = nf.cross_validation(df=nf_df.drop(columns=['real_date']), n_windows=n_test, step_size=1, val_size=n_val, refit=False)
    cv_df = cv_df.reset_index(drop=True)

    cv_df['ds'] = pd.to_datetime(cv_df['ds'])
    nf_df_dates = nf_df[['unique_id', 'ds', 'real_date']].copy()
    nf_df_dates['ds'] = pd.to_datetime(nf_df_dates['ds'])
    cv_df = pd.merge(cv_df, nf_df_dates, on=['unique_id', 'ds'], how='left')

    cv_df.to_csv(cv_file_path, index=False)
    print(f"✅ Prediksi tersimpan ke {cv_file_path}")

# VERIFIKASI STATUS EARLY STOPPING (Tetap jalan untuk membaca log jika ada)
stop_results = []
for m in models_list:
    m_name = m.__class__.__name__
    metrics_files = glob.glob(f'/kaggle/working/logs/{m_name}_V4/*/metrics.csv')
    if len(metrics_files) > 0:
        metrics_df = pd.read_csv(metrics_files[-1])
        if 'step' in metrics_df.columns:
            stop_step = metrics_df['step'].dropna().max()
            val_df = metrics_df.dropna(subset=['valid_loss'])
            min_val = val_df['valid_loss'].min() if not val_df.empty else None
            min_step = val_df.loc[val_df['valid_loss'].idxmin(), 'step'] if not val_df.empty else None
            last_val = val_df['valid_loss'].iloc[-1] if not val_df.empty else None
            status = "Max Steps Reached" if stop_step >= (MAX_STEPS - 1) else "Early Stopped"
            stop_results.append({'Model': m_name, 'Status': status, 'Stopped_at_Step': stop_step, 
                                 'Last_Val_Loss': last_val, 'Min_Val_Loss': min_val, 'Min_Val_Step': min_step})

print("=== STATUS EARLY STOPPING 7 MODEL (V4) ===")
display(pd.DataFrame(stop_results))


Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42


✅ MENDETEKSI FILE /kaggle/working/CV_Predictions_V4.csv SUDAH ADA!
Melewati proses training ulang... Langsung memuat data ke memori.
=== STATUS EARLY STOPPING 7 MODEL (V4) ===


,Model,Status,Stopped_at_Step,Last_Val_Loss,Min_Val_Loss,Min_Val_Step
0,TFT,Max Steps Reached,1999,0.000733,0.000731,1399
1,LSTM,Max Steps Reached,1999,0.000739,0.000723,49
2,NHITS,Max Steps Reached,1999,0.001028,0.001012,149
3,TiDE,Max Steps Reached,1999,0.000583,0.000582,599
4,NBEATSx,Max Steps Reached,1999,0.001002,0.000979,99
5,TSMixerx,Max Steps Reached,1999,0.000730,0.000672,199
6,KAN,Max Steps Reached,1999,0.000693,0.000693,1999


In [10]:
# ==========================================
# EKSTRAKSI FEATURE IMPORTANCE TFT (V4 FINAL)
# ==========================================
import os
import pandas as pd
import numpy as np

try:
    print("Mengekstrak model TFT secara mandiri untuk menghindari bug NeuralForecast state...")
    from neuralforecast.models import TFT
    from neuralforecast import NeuralForecast
    
    tft_only = TFT(h=H, input_size=INPUT_SIZE, hist_exog_list=exog_cols, 
                   max_steps=MAX_STEPS, val_check_steps=VAL_CHECK_STEPS, 
                   early_stop_patience_steps=CHOSEN_PATIENCE, random_seed=SEED, scaler_type='standard')
    
    # KUNCI UTAMA: Paksa pakai 1 GPU saja agar tidak bentrok (DDP Crash) di Kaggle T4x2
    tft_only.trainer_kwargs = {'devices': 1}
    
    nf_tft = NeuralForecast(models=[tft_only], freq='D')
    
    print("Melatih TFT tunggal (membutuhkan waktu ~2 menit)...")
    nf_tft.fit(df=df_train.drop(columns=['real_date']), val_size=n_val)
    
    _ = nf_tft.predict(df=df_test.drop(columns=['real_date']))
    
    tft_model = nf_tft.models[0]
    importances = tft_model.feature_importances()
    
    if isinstance(importances, dict) and 'Past variable importance over time' in importances:
        df_raw = importances['Past variable importance over time']
        mean_imp = df_raw.mean().reset_index()
        mean_imp.columns = ['Feature', 'Mean_Importance']
        
        mean_imp = mean_imp[~mean_imp['Feature'].isin(['unique_id', 'ds', 'y', 'observed_target', 'repeated_target'])].copy()
        df_importance = mean_imp.sort_values(by='Mean_Importance', ascending=False).reset_index(drop=True)
    else:
        df_importance = importances.sort_values(by=importances.columns[-1], ascending=False).reset_index(drop=True)

    print("\n=== TFT FEATURE IMPORTANCE (V4 FINAL - LOG VOLATILITY) ===")
    display(df_importance)
    
    os.makedirs('/kaggle/working/results', exist_ok=True)
    df_importance.to_csv('/kaggle/working/results/TFT_Mean_Importance_V4.csv', index=False)
    print("\n✅ Tersimpan ke /kaggle/working/results/TFT_Mean_Importance_V4.csv")

except Exception as e:
    import traceback
    print(f"Gagal mengekstrak TFT Feature Importance: {type(e).__name__} - {e}")
    traceback.print_exc()


Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores


Mengekstrak model TFT secara mandiri untuk menghindari bug NeuralForecast state...
Melatih TFT tunggal (membutuhkan waktu ~2 menit)...


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                    ┃ Type                     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss                    │ MAE                      │      0 │ train │     0 │
│ 1 │ hist_cat_embeddings     │ ModuleList               │      0 │ train │     0 │
│ 2 │ futr_cat_embeddings     │ ModuleList               │      0 │ train │     0 │
│ 3 │ stat_cat_embeddings     │ ModuleList               │      0 │ train │     0 │
│ 4 │ padder_train            │ ConstantPad1d            │      0 │ train │     0 │
│ 5 │ scaler                  │ TemporalNorm             │      0 │ train │     0 │
│ 6 │ embedding               │ TFTEmbedding             │  5.6 K │ train │     0 │
│ 7 │ temporal_encoder        │ TemporalCovariateEncoder │  2.3 M │ train │     0 │
│ 8 │ temporal_fusion_decoder │ TemporalFusionDecoder    │  256 K │ train │     0 │
│ 9 │ output_adapter          │ Linear                   │    129 │ train │     0 │
└───┴─────────────────────────┴──────────────────────────┴────────┴───────┴───────┘

Trainable params: 2.6 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 2.6 M                                                                                                
Total estimated model params size (MB): 10.383                                                                     
Modules in train mode: 251                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

... (4 outputs left out)

/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.



=== TFT FEATURE IMPORTANCE (V4 FINAL - LOG VOLATILITY) ===


,Feature,Mean_Importance
0,net_spill_saham_tsla,0.135668
1,ret_crypto_eth_usd,0.095977
2,net_spill_crypto_btc_usd,0.087566
3,net_spill_saham_nvda,0.083602
4,net_spill_crypto_eth_usd,0.052459
5,net_spill_saham_meta,0.049021
6,spillover_ai_to_btcidr,0.043504
7,net_spill_saham_googl,0.042990
8,tvp_tsi,0.042335
9,ret_saham_nvda,0.033335



✅ Tersimpan ke /kaggle/working/results/TFT_Mean_Importance_V4.csv


In [11]:
# ==========================================
# EVALUASI GARCH, EGARCH, & DM TEST
# ==========================================
from arch import arch_model
import time
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats import norm

returns_pct = df['ret_crypto_btc_idr_indodax'] * 100
train_returns = returns_pct.iloc[:n_train+n_val]
test_returns = returns_pct.iloc[n_train+n_val:]

garch_h1, egarch_h1 = [], []
garch_h10, egarch_h10 = [], []

print(f"Memulai Rolling Forecast GARCH/EGARCH (Total iterasi: {len(test_returns)})...")
start_time = time.time()
for i in range(len(test_returns)):
    window = returns_pct.iloc[:n_train+n_val+i]
    m_garch = arch_model(window, vol='GARCH', p=1, q=1, dist='normal').fit(disp='off', show_warning=False)
    m_egarch = arch_model(window, vol='EGARCH', p=1, q=1, dist='normal').fit(disp='off', show_warning=False, options={'maxiter': 1000})
    
    f_garch = m_garch.forecast(horizon=10, reindex=False)
    f_egarch = m_egarch.forecast(horizon=10, reindex=False, method='simulation', simulations=1000)
    
    garch_h1.append(f_garch.variance.values[-1, 0] / (100**2))
    egarch_h1.append(f_egarch.variance.values[-1, 0] / (100**2))
    garch_h10.append(f_garch.variance.values[-1, 9] / (100**2))
    egarch_h10.append(f_egarch.variance.values[-1, 9] / (100**2))

var_cap = np.percentile(df_train['y'].values, 99.9)
egarch_h1 = np.clip(egarch_h1, 0, var_cap)
egarch_h10 = np.clip(egarch_h10, 0, var_cap)

# FIX MERGE: Pastikan real_dates bertipe DatetimeIndex yang sinkron dengan cv_df
if isinstance(test_returns.index, pd.DatetimeIndex):
    real_dates = test_returns.index
else:
    real_dates = pd.to_datetime(df['Date'].iloc[n_train+n_val:].values if 'Date' in df.columns else df.index[n_train+n_val:])

df_garch = pd.DataFrame({
    'real_date': pd.to_datetime(real_dates), 
    'GARCH11_h1': garch_h1, 
    'EGARCH_h1': egarch_h1, 
    'GARCH11_h10': garch_h10, 
    'EGARCH_h10': egarch_h10
})

df_h1 = cv_df.groupby('cutoff').nth(0).reset_index()
df_h10 = cv_df.groupby('cutoff').nth(9).reset_index()

df_h1['real_date'] = pd.to_datetime(df_h1['real_date'])
df_h10['real_date'] = pd.to_datetime(df_h10['real_date'])

df_h1 = pd.merge(df_h1, df_garch[['real_date', 'GARCH11_h1', 'EGARCH_h1']], on='real_date', how='left').rename(columns={'GARCH11_h1': 'GARCH11', 'EGARCH_h1': 'EGARCH'})
df_h10 = pd.merge(df_h10, df_garch[['real_date', 'GARCH11_h10', 'EGARCH_h10']], on='real_date', how='left').rename(columns={'GARCH11_h10': 'GARCH11', 'EGARCH_h10': 'EGARCH'})

print("Sanity Check GARCH Merge (Monotonic):", df_h1['real_date'].is_monotonic_increasing)

full_y_by_date = nf_df.set_index('ds')['y'].sort_index()
def build_persistence(d):
    y_persist = (pd.to_datetime(d['ds']) - pd.Timedelta(days=1)).map(full_y_by_date)
    return y_persist.fillna(d['y']).values
    
for d in [df_h1, df_h10]:
    d['y_persist'] = build_persistence(d)
    d['y_hist_mean'] = df_train['y'].mean()
    d.replace([np.inf, -np.inf], np.nan, inplace=True)

def evaluate_models(df_subset):
    all_models = ['TFT', 'LSTM', 'NHITS', 'TiDE', 'NBEATSx', 'TSMixerx', 'KAN', 'GARCH11', 'EGARCH']
    results = []
    for m in all_models:
        mask = df_subset['y'].notna() & df_subset[m].notna()
        d_ev = df_subset[mask]
        if len(d_ev) < 2: continue
        y_true, y_pred = d_ev['y'].values, d_ev[m].values
        r2 = r2_score(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mae = mean_absolute_error(y_true, y_pred)
        t_diff = np.diff(y_true)
        p_diff = y_pred[1:] - y_true[:-1]
        dir_acc = np.mean(np.sign(t_diff) == np.sign(p_diff)) * 100 if len(t_diff) > 0 else np.nan
        results.append({'Model': m, 'N': len(y_true), 'R2': r2, 'RMSE': rmse, 'MAE': mae, 'DirAcc (%)': dir_acc})
    return pd.DataFrame(results)

print("=== METRIK h=1 (V4) ===")
display(evaluate_models(df_h1).round(6))
print("\n=== METRIK h=10 (V4) ===")
display(evaluate_models(df_h10).round(6))

def run_dm(df_subset, lag, title):
    res, alpha = [], 0.05 / 18
    for m in ['TFT', 'LSTM', 'NHITS', 'TiDE', 'NBEATSx', 'TSMixerx', 'KAN', 'GARCH11', 'EGARCH']:
        for base_col, base_name in [('y_persist', 'Persistence'), ('y_hist_mean', 'Historical Mean')]:
            mask = df_subset['y'].notna() & df_subset[m].notna() & df_subset[base_col].notna()
            yt, ym, yb = df_subset.loc[mask, 'y'].values, df_subset.loc[mask, m].values, df_subset.loc[mask, base_col].values
            T = len(yt)
            if T < max(3, lag + 2): 
                res.append({'Model': m, 'Baseline': base_name, 'N': T, 'DM Statistic': 0, 'p-value mentah': 1.0, 'Signifikan (Bonferroni)?': False}); continue
            d = (yt - yb)**2 - (yt - ym)**2
            d_mean = np.mean(d)
            nw_var = np.var(d, ddof=0)
            for k in range(1, lag + 1):
                nw_var += 2 * (1 - k / (lag + 1)) * np.cov(d[:-k], d[k:])[0, 1]
            if nw_var <= 0:
                res.append({'Model': m, 'Baseline': base_name, 'N': T, 'DM Statistic': 0, 'p-value mentah': 1.0, 'Signifikan (Bonferroni)?': False}); continue
            DM_stat = d_mean / np.sqrt(nw_var / T)
            p_val = 2 * (1 - norm.cdf(abs(DM_stat)))
            res.append({'Model': m, 'Baseline': base_name, 'N': T, 'DM Statistic': DM_stat, 'p-value mentah': p_val, 'Signifikan (Bonferroni)?': bool(p_val < alpha)})
    print(f"\n=== {title} ===")
    display(pd.DataFrame(res))

run_dm(df_h1, lag=0, title="DM TEST h=1 (lag=0) V4")
run_dm(df_h10, lag=9, title="DM TEST h=10 (lag=9) V4")


Memulai Rolling Forecast GARCH/EGARCH (Total iterasi: 335)...


/usr/local/lib/python3.13/dist-packages/arch/univariate/volatility.py:2838: RuntimeWarning: overflow encountered in exp
  paths[loc, :, :] = np.exp(_lnsigma2[:, m:])
/usr/local/lib/python3.13/dist-packages/arch/univariate/volatility.py:2838: RuntimeWarning: overflow encountered in exp
  paths[loc, :, :] = np.exp(_lnsigma2[:, m:])
/usr/local/lib/python3.13/dist-packages/arch/univariate/volatility.py:2838: RuntimeWarning: overflow encountered in exp
  paths[loc, :, :] = np.exp(_lnsigma2[:, m:])
/usr/local/lib/python3.13/dist-packages/arch/univariate/volatility.py:2838: RuntimeWarning: overflow encountered in exp
  paths[loc, :, :] = np.exp(_lnsigma2[:, m:])


Sanity Check GARCH Merge (Monotonic): True
=== METRIK h=1 (V4) ===


,Model,N,R2,RMSE,MAE,DirAcc (%)
0,TFT,335,-0.433354,0.001843,0.000699,67.365269
1,LSTM,335,-0.115666,0.001626,0.000595,67.964072
2,NHITS,335,-0.257777,0.001726,0.000767,54.191617
3,TiDE,335,-0.056524,0.001582,0.000488,70.958084
4,NBEATSx,335,-0.340562,0.001782,0.000795,52.994012
5,TSMixerx,335,-0.343860,0.001784,0.000681,68.562874
6,KAN,335,-0.114597,0.001625,0.000569,71.257485
7,GARCH11,326,-0.076623,0.001617,0.000750,65.230769
8,EGARCH,326,-62.049941,0.012378,0.005991,58.769231



=== METRIK h=10 (V4) ===


,Model,N,R2,RMSE,MAE,DirAcc (%)
0,TFT,335,-0.308460,0.001762,0.000636,64.371257
1,LSTM,335,-0.145034,0.001648,0.000598,71.257485
2,NHITS,335,-0.850094,0.002095,0.000892,63.772455
3,TiDE,335,-0.049785,0.001578,0.000495,70.658683
4,NBEATSx,335,-0.900381,0.002124,0.000908,60.479042
5,TSMixerx,335,-0.172242,0.001668,0.000637,68.862275
6,KAN,335,-0.090200,0.001608,0.000582,71.556886
7,GARCH11,335,-0.108339,0.001622,0.000813,64.371257
8,EGARCH,331,-58.943296,0.011994,0.005692,58.484848



=== DM TEST h=1 (lag=0) V4 ===


,Model,Baseline,N,DM Statistic,p-value mentah,Signifikan (Bonferroni)?
0,TFT,Persistence,335,-0.644396,5.193189e-01,False
1,TFT,Historical Mean,335,-1.069021,2.850601e-01,False
2,LSTM,Persistence,335,0.321942,7.474963e-01,False
3,LSTM,Historical Mean,335,1.248284,2.119270e-01,False
4,NHITS,Persistence,335,-0.325272,7.449750e-01,False
5,NHITS,Historical Mean,335,-0.203122,8.390399e-01,False
6,TiDE,Persistence,335,0.522451,6.013561e-01,False
7,TiDE,Historical Mean,335,2.064749,3.894672e-02,False
8,NBEATSx,Persistence,335,-0.666763,5.049238e-01,False
9,NBEATSx,Historical Mean,335,-0.494338,6.210673e-01,False



=== DM TEST h=10 (lag=9) V4 ===


,Model,Baseline,N,DM Statistic,p-value mentah,Signifikan (Bonferroni)?
0,TFT,Persistence,335,-0.552548,0.580573,False
1,TFT,Historical Mean,335,-0.586708,0.557400,False
2,LSTM,Persistence,335,0.583652,0.559454,False
3,LSTM,Historical Mean,335,0.508209,0.611307,False
4,NHITS,Persistence,335,-1.682265,0.092517,False
5,NHITS,Historical Mean,335,-1.497556,0.134249,False
6,TiDE,Persistence,335,1.451957,0.146514,False
7,TiDE,Historical Mean,335,1.624714,0.104223,False
8,NBEATSx,Persistence,335,-1.721808,0.085104,False
9,NBEATSx,Historical Mean,335,-1.546087,0.122084,False


In [12]:
# ==========================================
# PENGECEKAN FILE FINAL
# ==========================================
import os
import pandas as pd

print("=== PENGECEKAN FILE FINAL ===")

files_to_check = [
    '/kaggle/working/CV_Predictions_V4.csv',
    '/kaggle/working/results/TFT_Mean_Importance_V4.csv'
]

semua_aman = True

for path in files_to_check:
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        print(f"✅ ADA: {path} ({size_kb:.2f} KB)")
        
        # Cek sekilas isi file
        df_check = pd.read_csv(path)
        print(f"   -> Shape: {df_check.shape}")
        
        # Cek khusus jika ada kolom kosong (NaN) di kolom prediksi
        if 'TFT' in df_check.columns:
            nan_count = df_check['TFT'].isna().sum()
            if nan_count > 0:
                print(f"   -> ⚠️ PERINGATAN: Ada {nan_count} baris NaN pada kolom TFT!")
                semua_aman = False
    else:
        print(f"❌ TIDAK ADA: {path} (Belum di-generate atau gagal!)")
        semua_aman = False

print("\n=============================")
if semua_aman:
    print("🚀 AMAN 100%! Silakan klik folder 'Output' (atau direktori /kaggle/working) di panel kanan Kaggle dan DOWNLOAD kedua file tersebut.")
else:
    print("⚠️ ADA MASALAH. Mohon cek pesan error di sel-sel sebelumnya.")


=== PENGECEKAN FILE FINAL ===
✅ ADA: /kaggle/working/CV_Predictions_V4.csv (491.93 KB)
   -> Shape: (3350, 12)
✅ ADA: /kaggle/working/results/TFT_Mean_Importance_V4.csv (0.60 KB)
   -> Shape: (20, 2)

🚀 AMAN 100%! Silakan klik folder 'Output' (atau direktori /kaggle/working) di panel kanan Kaggle dan DOWNLOAD kedua file tersebut.
